In [8]:
import os
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

# Find repo root by walking up until a 'src' directory exists
cwd = Path.cwd().resolve()
repo_root = cwd
for _ in range(6):
    if (repo_root / "src").exists():
        break
    if repo_root.parent == repo_root:
        break
    repo_root = repo_root.parent

# Fallback to workspace path
if not (repo_root / "src").exists():
    repo_root = Path(r"C:\Users\yosrk\DeepFake")

REPO_ROOT = repo_root
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Local utilities
try:
    from src.data.dataset import DeepfakeDataset
    from src.utils.fft_transform import compute_fft_magnitude
    IMPORT_OK = True
except Exception as e:
    IMPORT_OK = False
    print(f"Warning: could not import repo utilities: {e}")

print(f"FFT notebook environment prepared. REPO_ROOT={REPO_ROOT} IMPORT_OK={IMPORT_OK}")

FFT notebook environment prepared. REPO_ROOT=C:\Users\yosrk\DeepFake IMPORT_OK=False


In [12]:
# Inject a concrete sample image from the local Kaggle dataset for demo
FFT_SAMPLE_IMAGE = r"data\\processed\\processed\\FaceForensics++_C23\\real\\000\\frame_000000_face_00.jpg"
print('Using FFT_SAMPLE_IMAGE =', FFT_SAMPLE_IMAGE)

Using FFT_SAMPLE_IMAGE = data\\processed\\processed\\FaceForensics++_C23\\real\\000\\frame_000000_face_00.jpg


# 02 FFT Analysis
Analyze frequency artifacts from deepfake images and verify the FFT-stream assumptions used by the model.

This notebook is configured for local repo paths and optional environment variables:
- `FFT_SAMPLE_IMAGE` for a single image demo
- `FFT_CSV_PATH` and `FFT_ROOT_DIR` for dataset-based inspection

In [13]:
sample_image = resolve_sample_image()
if sample_image:
    plot_fft_for_image(sample_image, title="FFT analysis sample")
else:
    print("Set FFT_SAMPLE_IMAGE or FFT_CSV_PATH to render the FFT analysis demo.")

No FFT_SAMPLE_IMAGE found. To analyze a dataset sample, set FFT_SAMPLE_IMAGE env var to a valid image path.
Set FFT_SAMPLE_IMAGE or FFT_CSV_PATH to render the FFT analysis demo.


In [15]:
if 'FFT_SAMPLE_IMAGE' not in globals():
    FFT_SAMPLE_IMAGE = os.environ.get("FFT_SAMPLE_IMAGE")

FFT_ROOT_DIR = Path(os.environ.get("FFT_ROOT_DIR", str(REPO_ROOT)))


def resolve_sample_image() -> str | None:
    # 1) prefer explicit env/variable
    if FFT_SAMPLE_IMAGE and Path(FFT_SAMPLE_IMAGE).exists():
        return FFT_SAMPLE_IMAGE

    # 2) known Kaggle dataset location (fallback)
    known = REPO_ROOT / 'data' / 'processed' / 'processed' / 'FaceForensics++_C23' / 'real' / '000' / 'frame_000000_face_00.jpg'
    if known.exists():
        return str(known)

    # 3) No dataset introspection in this standalone notebook; instruct the user
    print("No FFT_SAMPLE_IMAGE found. To analyze a dataset sample, set FFT_SAMPLE_IMAGE env var to a valid image path.")
    return None

print(f"FFT root dir : {FFT_ROOT_DIR}")
print("Sample image :", resolve_sample_image())

FFT root dir : C:\Users\yosrk\DeepFake
No FFT_SAMPLE_IMAGE found. To analyze a dataset sample, set FFT_SAMPLE_IMAGE env var to a valid image path.
Sample image : None


In [16]:
# Debug: show known path and existence
known = REPO_ROOT / 'data' / 'processed' / 'processed' / 'FaceForensics++_C23' / 'real' / '000' / 'frame_000000_face_00.jpg'
print('known:', known)
print('exists:', known.exists())
print('absolute:', known.resolve())

known: C:\Users\yosrk\DeepFake/data/processed/processed/FaceForensics++_C23/real/000/frame_000000_face_00.jpg
exists: False
absolute: /content/C:\Users\yosrk\DeepFake/data/processed/processed/FaceForensics++_C23/real/000/frame_000000_face_00.jpg


In [9]:
def compute_fft_magnitude(tensor: torch.Tensor) -> torch.Tensor:
    """Compute per-channel FFT magnitude for input tensor in [0,1].
    Accepts (B,C,H,W) or (C,H,W) and returns same spatial dims with channel axis preserved.
    """
    x = tensor
    if x.dim() == 3:
        x = x.unsqueeze(0)
    # x: (B,C,H,W)
    B, C, H, W = x.shape
    # Move to complex via FFT over last two dims
    fft = torch.fft.fft2(x, dim=(-2, -1))
    fft_shift = torch.roll(fft, shifts=(H // 2, W // 2), dims=(-2, -1))
    mag = torch.abs(fft_shift)
    # Log scale for visualization stability
    mag = torch.log1p(mag)
    return mag


def plot_fft_for_image(image_path: str, title: str | None = None):
    """Load an RGB image, compute its FFT magnitude, and display both views."""
    with Image.open(image_path) as img:
        rgb_image = img.convert("RGB")

    rgb_np = np.array(rgb_image)
    rgb_tensor = torch.from_numpy(rgb_np).permute(2, 0, 1).float() / 255.0
    fft_tensor = compute_fft_magnitude(rgb_tensor.unsqueeze(0)).squeeze(0)

    spatial = rgb_np
    if fft_tensor.ndim == 3 and fft_tensor.shape[0] >= 3:
        fft_vis = fft_tensor[:3].permute(1, 2, 0).cpu().numpy()
    else:
        fft_vis = fft_tensor.squeeze().cpu().numpy()

    fft_vis = fft_vis - np.min(fft_vis)
    fft_vis = fft_vis / (np.max(fft_vis) + 1e-8)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(spatial)
    axes[0].set_title(title or Path(image_path).name)
    axes[0].axis("off")

    axes[1].imshow(fft_vis, cmap="magma")
    axes[1].set_title("FFT magnitude")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

print("FFT plotting helper ready.")

FFT plotting helper ready.
